In [7]:
#1124 直線NP

# --- 1. SETUP AND IMPORTS ---
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg
import time
# Clean up
try:
    camera.unobserve_all()
    camera.stop()
except: pass
try:
    robot.stop()
except: pass

robot = Robot()
camera = Camera.instance()

# --- 2. GLOBAL VARIABLES (MEMORY) ---
last_left_speed = 0.0
last_right_speed = 0.0
no_line_counter = 0
MAX_COAST_FRAMES = 15  # Coast for ~0.5 seconds before stopping

last_trigger_time = 0 
TRIGGER_COOLDOWN =3.0

# --- 3. WIDGETS ---
feed_widget = widgets.Image(format='jpeg', width=300, height=300)
mask_widget = widgets.Image(format='jpeg', width=300, height=300)

# UPDATED COLOR DEFAULTS
# Increased Saturation Min to 60 to ignore the WHITE lines and GLARE
h_slider = widgets.IntRangeSlider(value=[20, 40], min=0, max=179, description='Hue')
s_slider = widgets.IntRangeSlider(value=[60, 255], min=0, max=255, description='Sat')
v_slider = widgets.IntRangeSlider(value=[100, 255], min=0, max=255, description='Val')

# CONTROL SLIDERS
speed_slider = widgets.FloatSlider(min=0.0, max=0.4, step=0.01, value=0.15, description='Base Speed')
turn_gain_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.25, description='Turn Strength')
dead_zone_slider = widgets.FloatSlider(min=0.0, max=0.5, step=0.01, value=0.05, description='Dead Zone')

status_label = widgets.HTML(value="Waiting...")

# --- 4. LOGIC WITH COASTING ---

def update_robot(change):
    global last_left_speed, last_right_speed, no_line_counter,last_trigger_time 

    frame = change['new']
    if frame is None: 
        return

    height, width, _ = frame.shape

    # ROI = 下半部影像
    roi = frame[int(height/2):height, 0:width] 
    
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    # ================================
    # 🔴 紅球（障礙物）偵測 → 執行固定動作
    # ================================
    red_lower = np.array([0, 0, 120])
    red_upper = np.array([100, 80, 255])

    mask_red = cv2.inRange(roi, red_lower, red_upper)
    contours_red, _ = cv2.findContours(mask_red, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    obstacle_centroid = None

    if len(contours_red) > 0:
        cnt = max(contours_red, key=cv2.contourArea)
        area = cv2.contourArea(cnt)

        if area > 100:    
            M = cv2.moments(cnt)
            if M['m00'] > 0:
                cx_r = int(M['m10']/M['m00'])
                cy_r = int(M['m01']/M['m00'])
                obstacle_centroid = (cx_r, cy_r)

                # 顯示偵測結果
                cv2.circle(roi, (cx_r, cy_r), 10, (0,0,255), -1)

    # ================================
    # 🔥 一看到紅點 → 執行你指定的固定動作序列
    # ================================
    if obstacle_centroid is not None:
        now = time.time()
        if now - last_trigger_time < TRIGGER_COOLDOWN:
            return  # 冷卻中 → 不觸發動作

        last_trigger_time = now


        print("🔴 Red object detected! Executing fixed movement sequence...")

#         robot.left_motor.value = 0.15
#         robot.right_motor.value = 0.4
#         time.sleep(0.4)

#         robot.left_motor.value = 0.2
#         robot.right_motor.value = 0.2
#         time.sleep(0.4)

#         robot.left_motor.value = 0.4
#         robot.right_motor.value = 0.15
#         time.sleep(0.8)

#         robot.left_motor.value = 0.2
#         robot.right_motor.value = 0.2
#         time.sleep(0.2)
        robot.set_motors(-0.3, -0.3); time.sleep(0.3)
        robot.set_motors(0.4, -0.1);  time.sleep(0.25)
        robot.set_motors(0.3, 0.3);   time.sleep(0.3)
        robot.set_motors(0.1, 0.4);   time.sleep(0.2)
        robot.set_motors(0, 0);       time.sleep(1)
        robot.set_motors(0.1, 0.3);   time.sleep(0.2)
#         robot.set_motors(0.3, 0.3);   time.sleep(0.3)

        robot.stop()
        print("✅ Movement done. Returning to line following.")
        return   # ← 重要！避免執行到尋線邏輯
    # ======================================================
    # 3. (原本) 黃線尋跡追蹤（保持不變）
    # ======================================================
    lower = np.array([h_slider.value[0], s_slider.value[0], v_slider.value[0]])
    upper = np.array([h_slider.value[1], s_slider.value[1], v_slider.value[1]])
    mask = cv2.inRange(hsv, lower, upper)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid_cnts = [c for c in contours if cv2.contourArea(c) > 100]
    valid_cnts.sort(key=cv2.contourArea, reverse=True)

    target_x = None

    if len(valid_cnts) >= 2:
        M1 = cv2.moments(valid_cnts[0])
        M2 = cv2.moments(valid_cnts[1])
        if M1['m00'] > 0 and M2['m00'] > 0:
            cx1 = int(M1['m10'] / M1['m00'])
            cx2 = int(M2['m10'] / M2['m00'])
            target_x = (cx1 + cx2) / 2
    elif len(valid_cnts) == 1:
        M1 = cv2.moments(valid_cnts[0])
        if M1['m00'] > 0:
            target_x = int(M1['m10'] / M1['m00'])

    # ---- 控制邏輯 ----
    if target_x is not None:
        no_line_counter = 0

        error = (target_x - (width / 2)) / (width / 2)
        base_speed = speed_slider.value
        steering = error * turn_gain_slider.value

        if abs(error) < dead_zone_slider.value:
            steering = 0.0
        
        adjusted_speed = base_speed - abs(steering) * 0.5
        if adjusted_speed < 0: adjusted_speed = 0

        left_motor = adjusted_speed + steering
        right_motor = adjusted_speed - steering

        left_motor = max(0.0, min(1.0, left_motor))
        right_motor = max(0.0, min(1.0, right_motor))

        robot.left_motor.value = left_motor
        robot.right_motor.value = right_motor

        last_left_speed = left_motor
        last_right_speed = right_motor

    else:
        no_line_counter += 1
        if no_line_counter < MAX_COAST_FRAMES:
            robot.left_motor.value = last_left_speed
            robot.right_motor.value = last_right_speed
        else:
            robot.stop()

    # Display
    feed_widget.value = bgr8_to_jpeg(roi)
    mask_widget.value = bgr8_to_jpeg(mask)



# --- START ---
camera.observe(update_robot, names='value')

ui = widgets.VBox([
    widgets.HBox([feed_widget, mask_widget]),
    status_label,
    widgets.Label("<b>Settings</b>"),
    h_slider, s_slider, v_slider,
    widgets.Label("<b>Control</b>"),
    speed_slider,
    turn_gain_slider,
    dead_zone_slider
])

display(ui)

In [6]:
# --- EMERGENCY STOP CELL ---

# Stop the robot's motors
robot.stop()

# Unlink the function from the camera to stop processing
camera.unobserve_all()

print("Robot and camera processing stopped.")

Robot and camera processing stopped.
